# Kalshi Trading Strategy Backtester

Kalshi is a prediction market. Contracts pay 1.00 dollar for Yes, 0.00 for No
#### A price of 30 cents means the market implies ~30% probability the event happens.

### Workflow
1. Authenticate with RSA API key
2. Fetch settled markets (resolved, outcomes known)
3. Fetch trade history
4. Backtest a strategy
5. Analyze results

## 1. Authentication

Kalshi uses **RSA request signing**. Every call requires:
1. Timestamp in milliseconds
2. Message = `timestamp + METHOD + path`
3. Sign with RSA private key using **PSS padding + SHA256**
4. Base64-encode signature
5. Send 3 headers: `KALSHI-ACCESS-KEY`, `KALSHI-ACCESS-SIGNATURE`, `KALSHI-ACCESS-TIMESTAMP`

In [2]:
import requests
import time
import base64
import pandas as pd
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding

KEY_ID           = 'XXXXx'
PRIVATE_KEY_PATH = '/private_key.pem'
BASE_URL         = 'https://api.elections.kalshi.com'


def get_auth_headers(method, path):
    """Returns the 3 signed headers Kalshi requires on every request."""
    with open(PRIVATE_KEY_PATH, 'rb') as f:
        private_key = serialization.load_pem_private_key(f.read(), password=None)
    timestamp = str(int(time.time() * 1000))          # milliseconds
    message   = (timestamp + method.upper() + path).encode('utf-8')
    # PSS padding is required — PKCS1v15 returns 401
    sig = private_key.sign(
        message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.DIGEST_LENGTH),
        hashes.SHA256()
    )
    return {
        'KALSHI-ACCESS-KEY':       KEY_ID,
        'KALSHI-ACCESS-SIGNATURE': base64.b64encode(sig).decode('utf-8'),
        'KALSHI-ACCESS-TIMESTAMP': timestamp,
        'Content-Type':            'application/json'
    }


# Sanity check — should print 200 and your balance
path = '/trade-api/v2/portfolio/balance'
r = requests.get(f'{BASE_URL}{path}', headers=get_auth_headers('GET', path))
print(r.status_code, r.json())

200 {'balance': 0, 'portfolio_value': 0, 'updated_ts': 1773164233}


## 2. Fetch Settled Markets

Settled markets have a `result` field (`yes` or `no`) so outcomes are known.

| Series | What it is |
|---|---|
| `KXNBAGAME`  | NBA game winner |
| `KXNHLGAME`  | NHL game winner |
| `KXNBATOTAL` | NBA total points |
| `KXNBAPTS`   | Player points prop |

Each game produces two contracts (one per side) — only one resolves YES.

In [3]:
SERIES = ['KXNBAGAME', 'KXNHLGAME', 'KXNBATOTAL', 'KXNBAPTS']


def fetch_settled_markets(series_list, limit=100):
    path    = '/trade-api/v2/markets'
    settled = []
    for series in series_list:
        r = requests.get(
            f'{BASE_URL}{path}',
            headers=get_auth_headers('GET', path),
            params={'limit': limit, 'series_ticker': series, 'status': 'settled'}
        )
        settled.extend(r.json().get('markets', []))
    return settled


settled_markets = fetch_settled_markets(SERIES)

# outcomes dict: ticker -> True (YES won) or False (NO won)
outcomes = {m['ticker']: (m['result'] == 'yes') for m in settled_markets}

print(f'Settled markets: {len(settled_markets)}')
print(f'YES: {sum(outcomes.values())}  NO: {sum(not v for v in outcomes.values())}')

pd.DataFrame([{'ticker': m['ticker'], 'title': m['title'],
               'result': m['result'], 'last_price': m.get('last_price')}
              for m in settled_markets]).head(10)

Settled markets: 400
YES: 191  NO: 209


,ticker,title,result,last_price
0,KXNBAGAME-26MAR09GSWUTA-UTA,Golden State at Utah Winner?,yes,99
1,KXNBAGAME-26MAR09GSWUTA-GSW,Golden State at Utah Winner?,no,1
2,KXNBAGAME-26MAR09NYKLAC-NYK,New York at Los Angeles C Winner?,no,1
3,KXNBAGAME-26MAR09NYKLAC-LAC,New York at Los Angeles C Winner?,yes,99
4,KXNBAGAME-26MAR09PHICLE-PHI,Philadelphia at Cleveland Winner?,no,1
5,KXNBAGAME-26MAR09PHICLE-CLE,Philadelphia at Cleveland Winner?,yes,99
6,KXNBAGAME-26MAR09DENOKC-OKC,Denver at Oklahoma City Winner?,yes,99
7,KXNBAGAME-26MAR09DENOKC-DEN,Denver at Oklahoma City Winner?,no,1
8,KXNBAGAME-26MAR09MEMBKN-MEM,Memphis at Brooklyn Winner?,no,1
9,KXNBAGAME-26MAR09MEMBKN-BKN,Memphis at Brooklyn Winner?,yes,99


## 3. Fetch Trade History

For each market we pull every executed trade. Each row has:
- `yes_price` — price in cents (0–100) = implied probability
- `count` — contracts traded
- `taker_side` — who initiated: `yes` or `no` buyer
- `created_time` — when the trade happened

This gives us the full **price history** of each market.

In [4]:
def fetch_trades(markets, max_markets=40):
    """Pull trade history. Capped at max_markets to avoid rate limits."""
    path       = '/trade-api/v2/markets/trades'
    all_trades = []
    for m in markets[:max_markets]:
        ticker = m['ticker']
        r = requests.get(
            f'{BASE_URL}{path}',
            headers=get_auth_headers('GET', path),
            params={'ticker': ticker, 'limit': 100}
        )
        trades = r.json().get('trades', [])
        for t in trades:
            t['ticker'] = ticker  # tag each trade with its market
        all_trades.extend(trades)
    df = pd.DataFrame(all_trades)
    if len(df):
        df['yes_price'] = pd.to_numeric(df['yes_price'])
        df['count']     = pd.to_numeric(df['count'])
    return df


trades_df = fetch_trades(settled_markets)
print(f'Trades: {len(trades_df)}  across  {trades_df["ticker"].nunique()} markets')
trades_df[['ticker', 'yes_price', 'count', 'taker_side', 'created_time']].head(10)

Trades: 4000  across  40 markets


,ticker,yes_price,count,taker_side,created_time
0,KXNBAGAME-26MAR09GSWUTA-UTA,99,254,no,2026-03-10T03:32:42.367299Z
1,KXNBAGAME-26MAR09GSWUTA-UTA,99,479,no,2026-03-10T03:32:42.277408Z
2,KXNBAGAME-26MAR09GSWUTA-UTA,99,66,no,2026-03-10T03:32:42.078561Z
3,KXNBAGAME-26MAR09GSWUTA-UTA,99,41,no,2026-03-10T03:32:41.945608Z
4,KXNBAGAME-26MAR09GSWUTA-UTA,99,31,no,2026-03-10T03:32:41.943451Z
5,KXNBAGAME-26MAR09GSWUTA-UTA,99,111,no,2026-03-10T03:32:41.63425Z
6,KXNBAGAME-26MAR09GSWUTA-UTA,99,84,no,2026-03-10T03:32:41.431527Z
7,KXNBAGAME-26MAR09GSWUTA-UTA,99,47,no,2026-03-10T03:32:40.85716Z
8,KXNBAGAME-26MAR09GSWUTA-UTA,99,738,no,2026-03-10T03:32:40.234876Z
9,KXNBAGAME-26MAR09GSWUTA-UTA,99,309,no,2026-03-10T03:32:40.032494Z


## 4. Backtester

Simulates buying and settling contracts against historical data.

- `buy_yes(ticker, price, n)` — spend `(price/100) x n` dollars
- `resolve(ticker, outcome)` — WIN: payout = `n x $0.93` (after 7% fee). LOSS: payout = $0
- `summary()` — print win rate, total PnL, ROI

**Kalshi fee:** 7% of winnings. A $1.00 win pays $0.93.

In [5]:
FEE = 0.07  # Kalshi charges 7% of winnings


class KalshiBacktester:
    def __init__(self, starting_capital=500):
        self.capital   = starting_capital
        self.starting  = starting_capital
        self.positions = {}   # ticker -> {contracts, cost}
        self.pnl_log   = []

    def buy_yes(self, ticker, price_cents, num_contracts):
        cost = (price_cents / 100) * num_contracts
        if cost > self.capital:
            return False
        self.capital -= cost
        pos = self.positions.setdefault(ticker, {'contracts': 0, 'cost': 0.0})
        pos['contracts'] += num_contracts
        pos['cost']      += cost
        return True

    def resolve(self, ticker, outcome_yes):
        if ticker not in self.positions:
            return 0
        pos    = self.positions.pop(ticker)
        payout = pos['contracts'] * (1.0 * (1 - FEE) if outcome_yes else 0.0)
        self.capital += payout
        pnl = payout - pos['cost']
        self.pnl_log.append({'ticker': ticker, 'pnl': pnl, 'won': outcome_yes})
        return pnl

    def summary(self):
        df        = pd.DataFrame(self.pnl_log)
        total_pnl = df['pnl'].sum() if len(df) else 0
        wins      = int((df['pnl'] > 0).sum()) if len(df) else 0
        n         = len(df)
        print(f'Trades resolved : {n}')
        print(f'Win / Loss      : {wins} / {n - wins}  ({100*wins/max(1,n):.1f}%)')
        print(f'Total PnL       : ${total_pnl:.2f}')
        print(f'ROI             : {100 * total_pnl / self.starting:.1f}%')
        print(f'Final capital   : ${self.capital:.2f}  (started ${self.starting:.2f})')
        return df


print('KalshiBacktester ready.')

KalshiBacktester ready.


## 5. Run Strategy: Buy Underdogs

**Idea:** Markets may underprice upsets. A team at 20 cents has implied 20% odds.
If the true win rate is higher, buying YES has positive expected value.

Tune `LOW` and `HIGH` to change the entry range.

In [6]:
# Strategy parameters — tune these
LOW          = 15   # buy YES when price >= LOW cents
HIGH         = 35   # buy YES when price <= HIGH cents
CONTRACTS    = 5    # contracts per trade
STARTING_CAP = 500  # starting bankroll ($)

bt = KalshiBacktester(starting_capital=STARTING_CAP)

# Scan every historical trade — enter when price is in our range
for _, row in trades_df.iterrows():
    ticker, price = row['ticker'], row['yes_price']
    if LOW <= price <= HIGH and ticker not in bt.positions:
        bt.buy_yes(ticker, price, CONTRACTS)

# Settle every open position against the known outcome
for ticker, resolved_yes in outcomes.items():
    if ticker in bt.positions:
        pnl = bt.resolve(ticker, resolved_yes)
        label = 'WIN ' if pnl > 0 else 'LOSS'
        print(f'{label}  {ticker:<55}  ${pnl:.2f}')

print()
results_df = bt.summary()


Trades resolved : 0
Win / Loss      : 0 / 0  (0.0%)
Total PnL       : $0.00
ROI             : 0.0%
Final capital   : $500.00  (started $500.00)


## 6. Visualize & Break-Even Analysis

**Break-even formula:** buying at price `p` cents with 7% fee:
`required win rate = p / (100 x 0.93)`

Example: buying at 20 cents needs `20/93 = 21.5%` win rate just to break even.

In [7]:
import plotly.express as px

if len(results_df):
    fig = px.bar(results_df, x='ticker', y='pnl', color='won',
                 title='PnL per Resolved Market',
                 labels={'pnl': 'Profit/Loss ($)'})
    fig.update_xaxes(tickangle=45)
    fig.show()

    results_df['cum_pnl'] = results_df['pnl'].cumsum()
    fig2 = px.line(results_df, y='cum_pnl', title='Cumulative PnL',
                   labels={'cum_pnl': 'Cumulative PnL ($)', 'index': 'Trade #'})
    fig2.show()

print('Break-even win rates by entry price:')
for p in [10, 15, 20, 25, 30, 35, 40]:
    print(f'  Buy at {p:2d}c  ->  need {p / (100 * (1 - FEE)) * 100:.1f}% win rate to break even')

Break-even win rates by entry price:
  Buy at 10c  ->  need 10.8% win rate to break even
  Buy at 15c  ->  need 16.1% win rate to break even
  Buy at 20c  ->  need 21.5% win rate to break even
  Buy at 25c  ->  need 26.9% win rate to break even
  Buy at 30c  ->  need 32.3% win rate to break even
  Buy at 35c  ->  need 37.6% win rate to break even
  Buy at 40c  ->  need 43.0% win rate to break even
